# Binadamu-Katika-Mzunguko: Mngato wa Kabla ya Hatua, Kiwango cha Hatari, na Kumbukumbu za Ukaguzi

README ya somo hili inatangaza Binadamu-Katika-Mzunguko kwa kipande kifupi kinachoomba mtumiaji `APPROVE` au `REJECT` baada ya wakala tayari kutoa jibu. Mfano huo ni mwanzo mzuri, lakini utekelezaji wa HITL wa uzalishaji kawaida huhitaji vipengele vitatu zaidi:

1. **Mngato wa kabla ya hatua** unaofanyika **kabla** wakala hajatekeleza hatua hatarishi, ili gharama, kutotendeka kinyume, na ucheleweshaji visidhibitiwe vibaya.
2. **Kugawanya hatari kwa ngazi**, ili hatua za hatari ya chini ziendeshwe moja kwa moja, hatua za hatari ya wastani ziridhishwe kwa batch, na hatua za hatari kubwa zitengemee mtu.
3. **Kumbukumbu za ukaguzi pamoja na mzunguko wa marekebisho**, ili kila uamuzi wa mngato urejodiwe kama JSONL, na kukataza kurejesha wakala na sababu ya muundo badala ya kuchapisha tu `Revising...`.

Kitabu hiki cha mazoezi hujenga kila moja ya haya juu ya msingi sawa kama `06-system-message-framework.ipynb`. Kinafanya mchakato mzima katika `DEMO_MODE = True` (bila haja ya pembejeo ya mwingiliano) au kwa anuani za kweli `input()` wakati `DEMO_MODE = False`. Kumbuka: katika DEMO_MODE jaribio la lengo la tatu limeandikwa kwa uandikaji hivyo mashine ya mzunguko yanaonekana mwisho hadi mwanzo. Urejeshaji wa kweli unaotegemea marekebisho unahitaji `DEMO_MODE = False` na mwendeshaji.

**Si sehemu ya somo (inatatuliwa katika masomo mengine):** uthibitishaji na udhibiti wa upatikanaji (Somo 06 README tishio 2), katikati ya simu ya zana (Somo 14 MAF uchunguzi wa kina), mifumo ya majadiliano ya mawakala wengi.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Mufano 1: Lango la awali la tendo

Kipande cha HITL cha README kinaita wakala kwanza, kisha kinaomba mtumiaji kuidhinisha matokeo. Hii ni mtiririko wa **baada ya tendo**. Wakala tayari ametekeleza, hivyo gharama ya wito wa LLM tayari imelipwa, na athari yoyote ya pembeni (barua pepe iliyotumwa, mstari wa database uliyoandikwa, maoni yaliyowekwa) tayari imekutokea.

Mtiririko wa **kabla ya tendo** huingiza lango kabla wakala hajafanya hatua hatari. Wakala anapendekeza tendo, lango linaamua kama litekelezeke, na athari ya pembeni hutokea tu baada ya idhini.

| Kipengele | Uidhinishaji wa baada ya tendo (kipande cha README) | Lango la kabla ya tendo (daftari hili) |
|---|---|---|
| Uidhinishaji hufanyika lini? | Baada ya wakala kutoa matokeo | Kabla ya athari yoyote ya pembeni kutekelezwa |
| Gharama ya LLM kwenye kukataa | Tayari imelipwa | Imelipwa tu kwa pendekezo, si tendo |
| Athari zisizoreversible | Inawezekana (tendo tayari limetokea) | Inazuia |
| Uwazi wa ukaguzi | Uidhinishaji ni tamko la kuchapisha | Uidhinishaji ni rekodi ya JSON yenye muda, tendo, sababu |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mfano 2: Kuainisha hatari kwa ngazi

Sio kila kitendo kinahitaji idhini ya binadamu. Kutafuta kwa kusoma tu dhidi ya API ya umma kuna hatari tofauti na kutuma barua pepe kwa mteja. Kutenda vitendo vyote kama vile vile huchafua umakini wa mtendakazi na kuchelewesha wakiendoaji.

Mfano rahisi wa ngazi 3:

| Ngazi | Mifano | Mtiririko wa Idhini |
|---|---|---|
| `chini` (kusoma tu) | Kutafuta katika hifadhidata ya maarifa, kuangalia chaguzi za ndege, kupata ukurasa wa wavuti wa umma | Endeshwa moja kwa moja, imerekodiwa kwa ukaguzi |
| `kati` (mabadiliko nafuu) | Kuhifadhi matokeo, kuongeza kielekezi, kupanga kumbusho | Endeshwa moja kwa moja, lakini hukusanywa kwa mapitio ya kila siku |
| `juu` (inayokabiliwa na watu au isiyoweza kurudishwa) | Kutuma barua pepe, kutoza kadi, kuchapisha kwenye kituo cha umma | Zuia hadi idhini ya binadamu ipatikane |

Hii ni aina moja ya kuainisha ngazi. Mifumo ya uzalishaji mara nyingi hutumia ngazi za kina zaidi (mfano, viwango vya ruhusa vya AWS IAM, ngazi za ufikiaji zinazoegemea majukumu). Toleo la ngazi 3 lililo hapa chini ni toleo ndogo kabisa linaloweza kutumika kwa mkiendoaji anayechanganya vitendo vya kusoma tu na vitendo vinavyosababisha athari.

Kichaguaji kilicho hapa chini kinatumia heuristics za maneno muhimu ili onyesho liendelee kuwa la lazima na gharama nafuu. Katika mfumo wa uzalishaji ungebadilisha hiki kwa kichaguaji kilichojifunza au mhandisi wa sera.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mfano 3: Rekodi ya ukaguzi na mzunguko wa marekebisho

`print("Response approved.")` si rekodi ya ukaguzi. Kwa imani, kila uamuzi wa lango unapaswa kurekodiwa kama tukio lililopangwa ambalo unaweza baadaye kuuliza, kuirudia, au kuambatisha kwenye ukaguzi wa tukio.

Sehemu mbili:

1. **JSONL ya kuongezwa tu.** Mstari mmoja kwa kila uamuzi, wenye alama ya wakati, hatua, ngazi, uamuzi, sababu. Rahisi kutafuta, rahisi kusafirisha kwa hifadhi halisi ya rekodi baadaye.
2. **Mzunguko wa marekebisho kwa kukataliwa.** Wakati lango linaporudisha `deny`, wakala hujitolea tena mwenyewe kwa sababu ya kukataliwa katika muktadha, ili pendekezo lijalo liweze kuepuka tatizo.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Rasilimali za ziada

Miradi mingine mingi ya umma hutekeleza mabadiliko ya mifumo hii ya HITL. Linganisha mbinu kupata inayofaa kwa stack yako:

- **LangChain** vifuniko vya zana za human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): vifuniko vya zana vinavyoacha utekelezaji kwa ajili ya maingizo ya binadamu.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ilibadilishwa hivi): hutumia nafasi ya wakala mahsusi kuwakilisha binadamu katika mazungumzo ya mawakala wengi.
- **Microsoft Agent Framework (MAF)** middleware ya mtendo wa kazi ([docs](https://learn.microsoft.com/agent-framework/)): middleware inayoendesha kila wito wa zana/kazi, inayofaa kwa mantiki ya kutawala na michakato ya idhini.

Kila mradi hushughulikia mifumo midogo mitatu kwa njia tofauti: LangChain huwaifunika kama zana, AutoGen hutumia nafasi ya wakala, na Microsoft Agent Framework hutumia middleware ya mtendo wa kazi. Soma utekelezaji mmoja au miwili kabisa kabla ya kuchagua muundo wa wakala wako mwenyewe.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
